# BAFU to ecoinvent activity matching audit

This notebook compares BAFU technosphere activities with ecoinvent activity and reference-product records. It is intentionally procedural and educational: each section performs one visible stage, prints interpretable statistics, and writes auditable outputs.

The matching policy is deterministic:

- Unit compatibility is mandatory.
- Reference product evidence is considered before activity-name evidence.
- Activity meaning and technical context can block or downgrade an otherwise similar text match.
- Geography is resolved only after the product/activity match is chosen.
- Obsolete BAFU activities whose names start with `xx` or `xxx` are excluded from the final matching structure and exported separately.

## 1. Imports and dependency checks

`rapidfuzz` is required because fuzzy matching quality is central to this audit. `sentence-transformers` and `rich` are optional. If semantic embeddings are unavailable, the notebook continues with deterministic text normalization, token logic, and rapidfuzz scores.

In [27]:
"""Imports and dependency checks for the BAFU to ecoinvent audit."""

from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import json
import math
import re
import sys
import warnings

import numpy as np
import pandas as pd

try:
    from rapidfuzz import fuzz, process
except ImportError as exc:
    raise SystemExit(
        "rapidfuzz is required for deterministic fuzzy matching. "
        "Install it in this notebook environment with: pip install rapidfuzz"
    ) from exc

try:
    from tqdm.auto import tqdm
except ImportError:
    print("tqdm is not installed. Progress bars are disabled.")

    def tqdm(iterable=None, **kwargs):
        """Fallback iterator when tqdm is not installed."""
        return iterable if iterable is not None else []

try:
    from rich.console import Console
    from rich.table import Table

    console = Console()
    RICH_AVAILABLE = True
except ImportError:
    console = None
    Table = None
    RICH_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer, util

    SENTENCE_TRANSFORMERS_AVAILABLE = True
except Exception:
    SentenceTransformer = None
    util = None
    SENTENCE_TRANSFORMERS_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

print("Imports complete.")
print(f"Optional rich output: {RICH_AVAILABLE}")
print(f"Optional sentence-transformers package: {SENTENCE_TRANSFORMERS_AVAILABLE}")

Imports complete.
Optional rich output: False
Optional sentence-transformers package: False


## 2. File paths and global variables

Run this notebook from the `clean bafu` folder. Change `REVIEW_MODE` to `"interactive"` if you want command-prompt review candidate selection. Keep it as `"export"` to complete without stopping and write all ambiguous candidates to Excel.

In [28]:
WORK_DIR = Path.cwd()
if not (WORK_DIR / "bafu_technosphere_flows.xlsx").exists() and (WORK_DIR / "clean bafu" / "bafu_technosphere_flows.xlsx").exists():
    WORK_DIR = WORK_DIR / "clean bafu"

BAFU_PATH = WORK_DIR / "bafu_technosphere_flows.xlsx"
ECOINVENT_PATH = WORK_DIR / "ECOINVENT ACTIVITY TECHNO.xlsx"

OUTPUT_WORKBOOK = WORK_DIR / "bafu_ecoinvent_activity_matching_audit.xlsx"
RUN_SUMMARY_JSON = WORK_DIR / "run_summary.json"
OBSOLETE_EXPORT = WORK_DIR / "obsolete_bafu_activities.xlsx"
REVIEW_CANDIDATES_EXPORT = WORK_DIR / "manual_review_candidates.xlsx"

REVIEW_MODE = "export"  # Change to "interactive" for command-prompt candidate review.
VALID_REVIEW_MODES = {"export", "interactive"}
if REVIEW_MODE not in VALID_REVIEW_MODES:
    raise ValueError(f"REVIEW_MODE must be one of {sorted(VALID_REVIEW_MODES)}")

TOP_PRODUCT_CANDIDATES = 16
TOP_ACTIVITY_CANDIDATES = 18
MAX_REVIEW_CANDIDATES_PER_ROW = 10

PRODUCT_FUZZY_ACCEPT_THRESHOLD = 92
ACTIVITY_ONLY_ACCEPT_THRESHOLD = 90
MIN_PRODUCT_REFERENCE_SCORE = 78
MIN_REVIEW_SCORE = 65

RUN_STARTED_AT = datetime.now().isoformat(timespec="seconds")

print(f"Working directory: {WORK_DIR}")
print(f"BAFU input: {BAFU_PATH}")
print(f"ecoinvent input: {ECOINVENT_PATH}")
print(f"Review mode: {REVIEW_MODE}")

Working directory: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu
BAFU input: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/bafu_technosphere_flows.xlsx
ecoinvent input: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/ECOINVENT ACTIVITY TECHNO.xlsx
Review mode: export


## 3. Expected columns, unit harmonization, and technical context terms

Unit harmonization is explicit. A row with an unmapped BAFU unit is blocked from matching and exported for review. Geography terms are not used to choose the product/activity match.

In [31]:
BAFU_SHEET = "BAFU"
ECOINVENT_SHEET = "Cut-Off AO"

BAFU_REQUIRED_COLUMNS = [
    "activity name",
    "activity uuid",
    "reference product",
    "product uuid",
    "activity unit",
    "geography",
]

ECOINVENT_REQUIRED_COLUMNS = [
    "Activity UUID & Product UUID",
    "Activity UUID",
    "ecoQuery URL",
    "Activity Name",
    "Geography",
    "Time Period",
    "Special Activity Type",
    "Sector",
    "ISIC Classification",
    "ISIC Section",
    "Product UUID",
    "Reference Product Name",
    "CPC Classification",
    "HS2017 Classification",
    "Unit",
    "Product Information",
    "CAS Number",
    "Cut-Off Classification",
]

UNIT_HARMONIZATION = {
    "kg": "kg",
    "kilogram": "kg",
    "kilograms": "kg",
    "g": "g",
    "gram": "g",
    "grams": "g",
    "m3": "m3",
    "m^3": "m3",
    "cubic meter": "m3",
    "cubic meters": "m3",
    "cubic metre": "m3",
    "cubic metres": "m3",
    "m2": "m2",
    "m^2": "m2",
    "square meter": "m2",
    "square meters": "m2",
    "square metre": "m2",
    "square metres": "m2",
    "m2*year": "m2*year",
    "m2 year": "m2*year",
    "m2-year": "m2*year",
    "square meter-year": "m2*year",
    "square metre-year": "m2*year",
    "km": "km",
    "kilometer": "km",
    "kilometers": "km",
    "kilometre": "km",
    "kilometres": "km",
    "km*year": "km*year",
    "km year": "km*year",
    "km-year": "km*year",
    "kilometer-year": "km*year",
    "kilometre-year": "km*year",
    "kwh": "kWh",
    "kilowatt hour": "kWh",
    "kilowatt-hour": "kWh",
    "mj": "MJ",
    "megajoule": "MJ",
    "megajoules": "MJ",
    "person*km": "person*km",
    "person km": "person*km",
    "person-kilometer": "person*km",
    "person kilometre": "person*km",
    "metric ton*km": "metric ton*km",
    "metric ton km": "metric ton*km",
    "ton kilometer": "metric ton*km",
    "ton-kilometer": "metric ton*km",
    "ton kilometre": "metric ton*km",
    "tonne kilometer": "metric ton*km",
    "tonne-kilometre": "metric ton*km",
    "ha": "ha",
    "hectare": "ha",
    "hectares": "ha",
    "m": "m",
    "meter": "m",
    "meters": "m",
    "metre": "m",
    "metres": "m",
    "m*year": "m*year",
    "m year": "m*year",
    "m-year": "m*year",
    "meter-year": "m*year",
    "metre-year": "m*year",
    "hour": "hour",
    "hours": "hour",
    "h": "hour",
    "unit": "unit",
    "units": "unit",
    "item": "unit",
    "year": "year",
    "years": "year",
    "a": "year",
}

FILLER_PHRASES_FOR_SCORING = [
    "at plant",
    "at mine",
    "at refinery",
    "at regional storehouse",
    "from petroleum refinery",
    "production mix",
    "product in",
]

DOMAIN_TERMS = {
    "chemical": [
        "acid",
        "chloride",
        "oxide",
        "hydroxide",
        "sulfate",
        "sulphate",
        "nitrate",
        "carbonate",
        "ammonia",
        "ethylene",
        "propylene",
        "benzene",
        "solvent",
        "polymer",
        "resin",
    ],
    "infrastructure": [
        "building",
        "road",
        "bridge",
        "airport",
        "railway",
        "network",
        "plant",
        "infrastructure",
        "pipeline",
    ],
    "energy": ["electricity", "heat", "steam", "power", "natural gas", "diesel", "fuel", "petrol", "oil"],
    "transport": ["transport", "freight", "lorry", "truck", "rail", "ship", "barge", "aircraft", "pipeline"],
    "waste": ["waste", "treatment", "recycling", "incineration", "landfill", "scrap", "defunct"],
    "metal": ["steel", "iron", "aluminium", "aluminum", "copper", "zinc", "nickel", "lead", "metal"],
    "mineral": ["gravel", "sand", "cement", "limestone", "clay", "gypsum", "stone", "mineral"],
    "paper": ["paper", "cardboard", "pulp", "packaging"],
    "agriculture": ["wheat", "maize", "barley", "milk", "meat", "soy", "crop", "seed", "fertilizer", "manure"],
    "food": ["food", "sugar", "oilseed", "vegetable", "fruit", "flour", "cheese", "beverage"],
    "service": ["service", "operation", "maintenance", "rental", "processing"],
}

CONTEXT_WARNING_RULES = [
    ("treatment", ["treatment", "treated"], "treatment_term_mismatch", 35),
    ("waste", ["waste", "scrap", "defunct", "end of life", "end-of-life"], "waste_context_warning", 35),
    ("market", ["market for", "market group for"], "market_activity_warning", 30),
    ("transport", ["transport", "freight", "lorry", "truck", "rail", "ship", "barge"], "transport_context_warning", 30),
    ("construction", ["construction", "constructing"], "broad_proxy_warning", 25),
    ("demolition", ["demolition", "dismantling"], "broad_proxy_warning", 35),
    ("recycling", ["recycling", "recycled"], "waste_context_warning", 35),
    ("incineration", ["incineration", "incinerator"], "waste_context_warning", 35),
    ("landfill", ["landfill"], "waste_context_warning", 35),
    ("operation", ["operation"], "broad_proxy_warning", 20),
    ("maintenance", ["maintenance"], "broad_proxy_warning", 20),
]

SERIOUS_CONTEXT_WARNINGS = {
    "unit_mismatch_blocked",
    "treatment_term_mismatch",
    "waste_context_warning",
    "transport_context_warning",
    "market_activity_warning",
}

## 4. Printing and dataframe inspection helpers

In [32]:
def cprint(message, style="bold cyan"):
    """Print a stage message with rich styling when available."""
    if RICH_AVAILABLE:
        console.print(message, style=style)
    else:
        print(message)


def as_text(value):
    """Convert missing values and scalars to a stripped string."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    return str(value).strip()


def missing_count(series):
    """Count missing or blank values in a pandas Series."""
    return int(series.isna().sum() + series.fillna("").astype(str).str.strip().eq("").sum())


def unique_count(series):
    """Count unique nonblank values in a pandas Series."""
    return int(series.fillna("").astype(str).str.strip().replace("", np.nan).dropna().nunique())


def show_preview(df, title, columns=None, max_rows=8):
    """Display a compact dataframe preview."""
    cprint(f"\n{title}", "bold green")
    if df.empty:
        print("No rows.")
        return
    preview = df.loc[:, columns] if columns else df
    display(preview.head(max_rows))


def print_key_statistics(df, label, key_columns):
    """Print row, column, missing-value, and uniqueness statistics."""
    cprint(f"\n{label}", "bold blue")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print("Missing values per key column:")
    for column in key_columns:
        print(f"  {column}: {missing_count(df[column]):,}")
    for column in key_columns:
        print(f"Unique {column}: {unique_count(df[column]):,}")

## 5. Load both Excel inputs and print raw statistics

In [33]:
def validate_columns(df, required_columns, label):
    """Validate that a dataframe contains all required columns."""
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def load_inputs(bafu_path, ecoinvent_path):
    """Load the two Excel workbooks and validate sheets and columns."""
    for path in [bafu_path, ecoinvent_path]:
        if not path.exists():
            raise FileNotFoundError(f"Required input file not found: {path}")

    bafu_xls = pd.ExcelFile(bafu_path)
    eco_xls = pd.ExcelFile(ecoinvent_path)

    if BAFU_SHEET not in bafu_xls.sheet_names:
        raise ValueError(f"BAFU workbook must contain sheet {BAFU_SHEET!r}")
    if ECOINVENT_SHEET not in eco_xls.sheet_names:
        raise ValueError(f"ecoinvent workbook must contain sheet {ECOINVENT_SHEET!r}")

    bafu = pd.read_excel(bafu_path, sheet_name=BAFU_SHEET, dtype="string")
    eco = pd.read_excel(ecoinvent_path, sheet_name=ECOINVENT_SHEET, dtype="string")

    validate_columns(bafu, BAFU_REQUIRED_COLUMNS, "BAFU workbook")
    validate_columns(eco, ECOINVENT_REQUIRED_COLUMNS, "ecoinvent workbook")

    metadata = {
        "bafu_sheet_names": bafu_xls.sheet_names,
        "ecoinvent_sheet_names": eco_xls.sheet_names,
    }
    return bafu, eco, metadata


def print_raw_statistics(bafu, eco, metadata):
    """Print raw workbook statistics requested for the audit."""
    cprint("Raw workbook sheet names", "bold magenta")
    print(f"BAFU sheets: {metadata['bafu_sheet_names']}")
    print(f"ecoinvent sheets: {metadata['ecoinvent_sheet_names']}")

    print_key_statistics(
        bafu,
        "Raw BAFU statistics",
        ["activity name", "activity uuid", "reference product", "product uuid", "activity unit", "geography"],
    )
    print(f"Unique BAFU activities: {unique_count(bafu['activity name']):,}")
    print(f"Unique BAFU reference products: {unique_count(bafu['reference product']):,}")
    print(f"Unique BAFU units: {unique_count(bafu['activity unit']):,}")
    print(f"Unique BAFU geographies: {unique_count(bafu['geography']):,}")

    print_key_statistics(
        eco,
        "Raw ecoinvent statistics",
        ["Activity Name", "Activity UUID", "Reference Product Name", "Product UUID", "Unit", "Geography"],
    )
    print(f"Unique ecoinvent activities: {unique_count(eco['Activity Name']):,}")
    print(f"Unique ecoinvent reference products: {unique_count(eco['Reference Product Name']):,}")
    print(f"Unique ecoinvent units: {unique_count(eco['Unit']):,}")
    print(f"Unique ecoinvent geographies: {unique_count(eco['Geography']):,}")

In [34]:
cprint("Stage: loading files", "bold cyan")
bafu_raw, eco_raw, workbook_metadata = load_inputs(BAFU_PATH, ECOINVENT_PATH)
print_raw_statistics(bafu_raw, eco_raw, workbook_metadata)
show_preview(bafu_raw, "BAFU preview")
show_preview(eco_raw, "ecoinvent preview")

Stage: loading files
Raw workbook sheet names
BAFU sheets: ['BAFU']
ecoinvent sheets: ['Cut-Off AO']

Raw BAFU statistics
Rows: 11,747
Columns: 6
Missing values per key column:
  activity name: 0
  activity uuid: 0
  reference product: 0
  product uuid: 0
  activity unit: 0
  geography: 2
Unique activity name: 8,017
Unique activity uuid: 11,747
Unique reference product: 8,017
Unique product uuid: 11,747
Unique activity unit: 16
Unique geography: 121
Unique BAFU activities: 8,017
Unique BAFU reference products: 8,017
Unique BAFU units: 16
Unique BAFU geographies: 121

Raw ecoinvent statistics
Rows: 26,533
Columns: 18
Missing values per key column:
  Activity Name: 0
  Activity UUID: 0
  Reference Product Name: 0
  Product UUID: 0
  Unit: 0
  Geography: 18
Unique Activity Name: 10,094
Unique Activity UUID: 22,993
Unique Reference Product Name: 4,346
Unique Product UUID: 4,346
Unique Unit: 18
Unique Geography: 334
Unique ecoinvent activities: 10,094
Unique ecoinvent reference products: 4,

,activity name,activity uuid,reference product,product uuid,activity unit,geography
0,"1,1-difluoroethane, HFC-152a, at plant",cda7d2ad-0df5-32cc-945b-49d9cb010895,"1,1-difluoroethane, HFC-152a, at plant",191327,kilogram,US
1,"1-butanol, propylene hydroformylation, at plant",b323f136-28be-3e86-bea4-e77610261147,"1-butanol, propylene hydroformylation, at plant",562971,kilogram,RER
2,"1-pentanol, at plant",7589a6ec-9584-3a34-974b-0ede8a255608,"1-pentanol, at plant",528491,kilogram,RER
3,"1-propanol, at plant",183b303d-d07b-3177-b091-c581e9c4ab20,"1-propanol, at plant",539705,kilogram,RER
4,"1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof",ec675697-6d86-3ed6-bbb6-430db22a50ee,"1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof",369027,unit,CH
5,"156 kWp flat-roof installation, multi-Si, on roof",7e86cb3d-68bc-3d47-98d9-4ea7726acc2e,"156 kWp flat-roof installation, multi-Si, on roof",230442,unit,CH
6,"156 kWp flat-roof installation, single-Si, on roof",7c5bcdfa-391e-388b-a2ec-2d6191d08437,"156 kWp flat-roof installation, single-Si, on roof",282327,unit,CH
7,"16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si",f343e106-7a73-3c54-b652-f574d1caea9b,"16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si",332989,unit,CH



ecoinvent preview


,Activity UUID & Product UUID,Activity UUID,ecoQuery URL,Activity Name,Geography,Time Period,Special Activity Type,Sector,ISIC Classification,ISIC Section,Product UUID,Reference Product Name,CPC Classification,HS2017 Classification,Unit,Product Information,CAS Number,Cut-Off Classification
0,3ebd3282-43d1-5d80-a8e3-b3dbc05e4e4e_4c2b1cc3-84e5-4e35-b74e-8815eadbc674,3ebd3282-43d1-5d80-a8e3-b3dbc05e4e4e,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/9766/documentation,[sulfonyl]urea-compound production,RER,2000 - 2025,ordinary transforming activity,Chemicals; Agriculture & Animal Husbandry,2021:Manufacture of pesticides and other agrochemical products,C - Manufacturing,4c2b1cc3-84e5-4e35-b74e-8815eadbc674,[sulfonyl]urea-compound,"34663: Herbicides, anti-sprouting products and plant-growth regulators","380893: Herbicides, anti-sprouting products and plant-growth regulators; other than containing goods of Subheading [...",kg,'[sulfonyl]urea-compound' represents an organic substance. The compounds fragment is a S-arylsulfonylurea funtional ...,<NA>,allocatable product
1,dc6ad157-446f-560c-b3c3-09c66be46dc7_4c2b1cc3-84e5-4e35-b74e-8815eadbc674,dc6ad157-446f-560c-b3c3-09c66be46dc7,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/3515/documentation,[sulfonyl]urea-compound production,RoW,2000 - 2025,ordinary transforming activity,Chemicals; Agriculture & Animal Husbandry,2021:Manufacture of pesticides and other agrochemical products,C - Manufacturing,4c2b1cc3-84e5-4e35-b74e-8815eadbc674,[sulfonyl]urea-compound,"34663: Herbicides, anti-sprouting products and plant-growth regulators","380893: Herbicides, anti-sprouting products and plant-growth regulators; other than containing goods of Subheading [...",kg,'[sulfonyl]urea-compound' represents an organic substance. The compounds fragment is a S-arylsulfonylurea funtional ...,<NA>,allocatable product
2,be43a1ca-9bdd-5400-b708-d3016a262262_1be4f7e4-5244-4f9d-b80d-7fbf1e337e2b,be43a1ca-9bdd-5400-b708-d3016a262262,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/1366/documentation,[thio]carbamate-compound production,RER,2000 - 2025,ordinary transforming activity,Chemicals; Agriculture & Animal Husbandry,2021:Manufacture of pesticides and other agrochemical products,C - Manufacturing,1be4f7e4-5244-4f9d-b80d-7fbf1e337e2b,[thio]carbamate-compound,"34663: Herbicides, anti-sprouting products and plant-growth regulators","380893: Herbicides, anti-sprouting products and plant-growth regulators; other than containing goods of Subheading [...",kg,'[thio]carbamate-compound' represents an organic substance. The compounds fragment is either a thiocarbamate ester o...,<NA>,allocatable product
3,b72cce10-8054-536b-9368-a1a5a62393d3_1be4f7e4-5244-4f9d-b80d-7fbf1e337e2b,b72cce10-8054-536b-9368-a1a5a62393d3,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/5329/documentation,[thio]carbamate-compound production,RoW,2000 - 2025,ordinary transforming activity,Chemicals; Agriculture & Animal Husbandry,2021:Manufacture of pesticides and other agrochemical products,C - Manufacturing,1be4f7e4-5244-4f9d-b80d-7fbf1e337e2b,[thio]carbamate-compound,"34663: Herbicides, anti-sprouting products and plant-growth regulators","380893: Herbicides, anti-sprouting products and plant-growth regulators; other than containing goods of Subheading [...",kg,'[thio]carbamate-compound' represents an organic substance. The compounds fragment is either a thiocarbamate ester o...,<NA>,allocatable product
4,b8af5656-7e33-5ce1-8e0a-8760fcc8a32a_20d0008c-e699-5377-9c04-64267728fb7f,b8af5656-7e33-5ce1-8e0a-8760fcc8a32a,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/276370/documentation,"1,1,1-trichloroethane production, vinyl chloride chlorination",GLO,2015 - 2025,ordinary transforming activity,Chemicals,2011:Manufacture of basic chemicals,C - Manufacturing,20d0008c-e699-5377-9c04-64267728fb7f,"1,1,1-trichloroethane","3411: Hydrocarbons and their halogenated, sulphonated, nitrated or nitrosated derivatives",2903: Halogenated deriva

## 6. Text cleaning, token logic, and unit normalization helpers

In [35]:
def clean_text(value, remove_fillers=True):
    """Normalize text for deterministic scoring."""
    text = as_text(value).lower()
    text = text.replace("&", " and ")
    text = text.replace("'", " ")
    text = re.sub(r"[\u2010-\u2015-]", " ", text)
    if remove_fillers:
        for phrase in FILLER_PHRASES_FOR_SCORING:
            text = text.replace(phrase, " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(value):
    """Return sorted word tokens used for context overlap."""
    stop_words = {
        "and",
        "or",
        "of",
        "for",
        "the",
        "a",
        "an",
        "to",
        "in",
        "from",
        "with",
        "by",
        "as",
    }
    return sorted(token for token in clean_text(value).split() if token and token not in stop_words)


def normalize_unit(value):
    """Map a BAFU or ecoinvent unit to a shared physical-unit label."""
    raw = as_text(value)
    if not raw:
        return None
    unit = raw.lower().strip()
    unit = unit.replace("²", "2").replace("³", "3")
    unit = unit.replace("·", "*").replace("/", " per ")
    unit = re.sub(r"\s+", " ", unit)
    if unit in UNIT_HARMONIZATION:
        return UNIT_HARMONIZATION[unit]
    compact = unit.replace("-", " ").replace("*", " ")
    compact = re.sub(r"\s+", " ", compact).strip()
    if compact in UNIT_HARMONIZATION:
        return UNIT_HARMONIZATION[compact]
    return None


def token_score(left, right):
    """Return a rapidfuzz token score from 0 to 100."""
    left_clean = clean_text(left)
    right_clean = clean_text(right)
    if not left_clean or not right_clean:
        return 0.0
    return float(fuzz.token_set_ratio(left_clean, right_clean))

## 7. Filter obsolete BAFU activities and export the audit list

In [36]:
def is_obsolete_bafu_activity(activity_name):
    """Return True when a BAFU activity name starts with xx or xxx."""
    name = as_text(activity_name).lower().strip()
    return name.startswith("xx") or name.startswith("xxx")


def filter_obsolete_bafu(bafu):
    """Split BAFU rows into active and obsolete records."""
    mask = bafu["activity name"].apply(is_obsolete_bafu_activity)
    obsolete = bafu.loc[mask].copy()
    active = bafu.loc[~mask].copy()
    return active, obsolete

In [37]:
cprint("Stage: filtering obsolete BAFU activities", "bold cyan")
bafu_active, obsolete_bafu = filter_obsolete_bafu(bafu_raw)
print(f"Rows before obsolete filter: {len(bafu_raw):,}")
print(f"Rows removed as obsolete: {len(obsolete_bafu):,}")
print(f"Rows retained for matching: {len(bafu_active):,}")

obsolete_bafu.to_excel(OBSOLETE_EXPORT, index=False)
print(f"Obsolete audit export written to: {OBSOLETE_EXPORT}")
show_preview(obsolete_bafu, "Obsolete BAFU preview")

Stage: filtering obsolete BAFU activities
Rows before obsolete filter: 11,747
Rows removed as obsolete: 1,289
Rows retained for matching: 10,458
Obsolete audit export written to: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/obsolete_bafu_activities.xlsx

Obsolete BAFU preview


,activity name,activity uuid,reference product,product uuid,activity unit,geography
10458,"xx 1,1-dimethylcyclopentane, from naphtha, at plant",95b9c55d-2fd8-3762-a056-14edfe924e75,"xx 1,1-dimethylcyclopentane, from naphtha, at plant",458833,kilogram,RER
10459,"xx 2,3-dimethylbutan, from naphtha, at plant",ea06c172-bcc0-3ad0-bd2c-7c0ff1a6da32,"xx 2,3-dimethylbutan, from naphtha, at plant",341653,kilogram,RER
10460,"xx 2,4-D, at regional storehouse",4e8daae6-b0ef-3926-a984-437e0a86471c,"xx 2,4-D, at regional storehouse",420346,kilogram,CH
10461,"xx 2-methyl-1-butanol, at plant",1780694e-0f44-3eee-8ffd-d2c2ce11b4f2,"xx 2-methyl-1-butanol, at plant",183501,kilogram,RER
10462,"xx 2-methylpentane, from naphtha, at plant",71511a4d-5f0d-3f14-aba4-4e506169e31f,"xx 2-methylpentane, from naphtha, at plant",531305,kilogram,RER
10463,"xx 3-methyl-1-butyl acetate, at plant",b0c01332-d778-3f14-89d5-2166fec8c294,"xx 3-methyl-1-butyl acetate, at plant",605402,kilogram,RER
10464,"xx 4-methyl-2-pentanone, at plant",6931fb2e-057d-3510-bdaa-539c0fd13f41,"xx 4-methyl-2-pentanone, at plant",212960,kilogram,RER
10465,"xx Acetic acid from acetaldehyde, at plant",c8788f81-09da-34fb-bf49-8d26c3218840,"xx Acetic acid from acetaldehyde, at plant",563567,kilogram,RER


## 8. Simplify BAFU to one row per activity, reference product, and unit

The final output later returns to the original non-obsolete geography rows. This simplification exists only to make the product/activity decision once per unique BAFU activity-reference product-unit combination.

In [38]:
def sorted_unique(values):
    """Return sorted unique nonblank string values."""
    cleaned = {as_text(value) for value in values if as_text(value)}
    return sorted(cleaned)


def pipe_join(values):
    """Join unique values with a pipe for readable Excel output."""
    return " | ".join(sorted_unique(values))


def make_simplified_bafu(bafu):
    """Group BAFU by activity name, reference product, and activity unit."""
    grouped = (
        bafu.groupby(["activity name", "reference product", "activity unit"], dropna=False)
        .agg(
            bafu_activity_uuids=("activity uuid", pipe_join),
            bafu_product_uuids=("product uuid", pipe_join),
            bafu_geographies=("geography", pipe_join),
            bafu_original_row_count=("activity name", "size"),
        )
        .reset_index()
    )
    grouped["normalized_bafu_unit"] = grouped["activity unit"].apply(normalize_unit)
    grouped["bafu_clean_activity"] = grouped["activity name"].apply(clean_text)
    grouped["bafu_clean_product"] = grouped["reference product"].apply(clean_text)
    return grouped

In [39]:
cprint("Stage: grouping BAFU", "bold cyan")
simplified_bafu = make_simplified_bafu(bafu_active)
print(f"Non-obsolete BAFU rows: {len(bafu_active):,}")
print(f"Simplified BAFU rows: {len(simplified_bafu):,}")
print(f"Unique simplified activities: {unique_count(simplified_bafu['activity name']):,}")
print(f"Unique simplified reference products: {unique_count(simplified_bafu['reference product']):,}")
print(f"Unique simplified units: {unique_count(simplified_bafu['activity unit']):,}")
print(f"Unique normalized BAFU units: {unique_count(simplified_bafu['normalized_bafu_unit']):,}")
print(f"Simplified rows with unmapped units: {simplified_bafu['normalized_bafu_unit'].isna().sum():,}")
show_preview(simplified_bafu, "Simplified BAFU preview")

Stage: grouping BAFU
Non-obsolete BAFU rows: 10,458
Simplified BAFU rows: 7,143
Unique simplified activities: 7,134
Unique simplified reference products: 7,134
Unique simplified units: 16
Unique normalized BAFU units: 16
Simplified rows with unmapped units: 0

Simplified BAFU preview


,activity name,reference product,activity unit,bafu_activity_uuids,bafu_product_uuids,bafu_geographies,bafu_original_row_count,normalized_bafu_unit,bafu_clean_activity,bafu_clean_product
0,"1,1-difluoroethane, HFC-152a, at plant","1,1-difluoroethane, HFC-152a, at plant",kilogram,cda7d2ad-0df5-32cc-945b-49d9cb010895,191327,US,1,kg,1 1 difluoroethane hfc 152a,1 1 difluoroethane hfc 152a
1,"1-butanol, propylene hydroformylation, at plant","1-butanol, propylene hydroformylation, at plant",kilogram,b323f136-28be-3e86-bea4-e77610261147,562971,RER,1,kg,1 butanol propylene hydroformylation,1 butanol propylene hydroformylation
2,"1-pentanol, at plant","1-pentanol, at plant",kilogram,7589a6ec-9584-3a34-974b-0ede8a255608,528491,RER,1,kg,1 pentanol,1 pentanol
3,"1-propanol, at plant","1-propanol, at plant",kilogram,183b303d-d07b-3177-b091-c581e9c4ab20,539705,RER,1,kg,1 propanol,1 propanol
4,"1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof","1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof",unit,ec675697-6d86-3ed6-bbb6-430db22a50ee,369027,CH,1,unit,1 3 mwp slanted roof installation multi si panel mounted on roof,1 3 mwp slanted roof installation multi si panel mounted on roof
5,"156 kWp flat-roof installation, multi-Si, on roof","156 kWp flat-roof installation, multi-Si, on roof",unit,7e86cb3d-68bc-3d47-98d9-4ea7726acc2e,230442,CH,1,unit,156 kwp flat roof installation multi si on roof,156 kwp flat roof installation multi si on roof
6,"156 kWp flat-roof installation, single-Si, on roof","156 kWp flat-roof installation, single-Si, on roof",unit,7c5bcdfa-391e-388b-a2ec-2d6191d08437,282327,CH,1,unit,156 kwp flat roof installation single si on roof,156 kwp flat roof installation single si on roof
7,"16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si","16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si",unit,f343e106-7a73-3c54-b652-f574d1caea9b,332989,CH,1,unit,16 667kwp 100m2 pv system for pvt installed on slanted roof single si,16 667kwp 100m2 pv system for pvt installed on slanted roof single si


## 9. Prepare ecoinvent and build deterministic indexes

In [41]:
def prepare_ecoinvent(eco):
    """Add normalized units and deterministic text fields to ecoinvent rows."""
    prepared = eco.copy()
    prepared["_ecoinvent_row_id"] = np.arange(len(prepared))
    prepared["_normalized_unit"] = prepared["Unit"].apply(normalize_unit)
    prepared["_clean_product"] = prepared["Reference Product Name"].apply(clean_text)
    prepared["_clean_activity"] = prepared["Activity Name"].apply(clean_text)
    prepared["_clean_combined"] = (
        prepared["Activity Name"].fillna("")
        + " "
        + prepared["Reference Product Name"].fillna("")
        + " "
        + prepared["Sector"].fillna("")
        + " "
        + prepared["Product Information"].fillna("")
    ).apply(clean_text)
    return prepared


def build_ecoinvent_indexes(eco):
    """Build grouped ecoinvent activity-product-unit indexes for matching."""
    prepared = prepare_ecoinvent(eco)
    mapped = prepared.loc[prepared["_normalized_unit"].notna()].copy()

    group_columns = [
        "Activity Name",
        "Reference Product Name",
        "Product UUID",
        "Unit",
        "_normalized_unit",
    ]
    aggregate_columns = {
        "Activity UUID": pipe_join,
        "Activity UUID & Product UUID": pipe_join,
        "ecoQuery URL": pipe_join,
        "Geography": pipe_join,
        "Time Period": pipe_join,
        "Special Activity Type": pipe_join,
        "Sector": pipe_join,
        "ISIC Classification": pipe_join,
        "ISIC Section": pipe_join,
        "CPC Classification": pipe_join,
        "HS2017 Classification": pipe_join,
        "Product Information": pipe_join,
        "CAS Number": pipe_join,
        "Cut-Off Classification": pipe_join,
        "_clean_product": "first",
        "_clean_activity": "first",
        "_clean_combined": pipe_join,
        "_ecoinvent_row_id": lambda values: "|".join(str(int(value)) for value in values),
    }
    groups = mapped.groupby(group_columns, dropna=False).agg(aggregate_columns).reset_index()
    groups["eco_group_id"] = np.arange(len(groups))

    prepared = prepared.merge(
        groups[group_columns + ["eco_group_id"]],
        on=group_columns,
        how="left",
    )

    product_choices_by_unit = {}
    activity_choices_by_unit = {}
    groups_by_unit_product = defaultdict(list)
    groups_by_unit_activity = defaultdict(list)

    for idx, row in groups.iterrows():
        unit = row["_normalized_unit"]
        product_key = row["_clean_product"]
        activity_key = row["_clean_activity"]
        groups_by_unit_product[(unit, product_key)].append(idx)
        groups_by_unit_activity[(unit, activity_key)].append(idx)

    for unit, unit_groups in groups.groupby("_normalized_unit"):
        product_choices_by_unit[unit] = sorted(unit_groups["_clean_product"].dropna().unique())
        activity_choices_by_unit[unit] = sorted(unit_groups["_clean_activity"].dropna().unique())

    indexes = {
        "product_choices_by_unit": product_choices_by_unit,
        "activity_choices_by_unit": activity_choices_by_unit,
        "groups_by_unit_product": groups_by_unit_product,
        "groups_by_unit_activity": groups_by_unit_activity,
    }
    return prepared, groups, indexes

In [42]:
cprint("Stage: normalizing units and building ecoinvent indexes", "bold cyan")
eco_prepared, eco_groups, eco_indexes = build_ecoinvent_indexes(eco_raw)
eco_unmapped_units = eco_prepared.loc[eco_prepared["_normalized_unit"].isna(), "Unit"].dropna().unique()

print(f"ecoinvent raw rows: {len(eco_raw):,}")
print(f"ecoinvent grouped activity-product-unit rows: {len(eco_groups):,}")
print(f"ecoinvent rows with unmapped units: {eco_prepared['_normalized_unit'].isna().sum():,}")
print(f"Normalized ecoinvent units: {sorted(eco_groups['_normalized_unit'].dropna().unique())}")
if len(eco_unmapped_units):
    print(f"Unmapped ecoinvent unit labels, for information only: {sorted(map(str, eco_unmapped_units))}")

unit_unmapped_simplified = simplified_bafu.loc[simplified_bafu["normalized_bafu_unit"].isna()].copy()
print(f"Simplified BAFU unit-unmapped rows blocked from matching: {len(unit_unmapped_simplified):,}")

Stage: normalizing units and building ecoinvent indexes
ecoinvent raw rows: 26,533
ecoinvent grouped activity-product-unit rows: 11,632
ecoinvent rows with unmapped units: 54
Normalized ecoinvent units: ['MJ', 'ha', 'hour', 'kWh', 'kg', 'km', 'km*year', 'm', 'm*year', 'm2', 'm2*year', 'm3', 'metric ton*km', 'person*km', 'unit']
Unmapped ecoinvent unit labels, for information only: ['guest night', 'kg*day', 'l']
Simplified BAFU unit-unmapped rows blocked from matching: 0


## 10. Scoring helpers

The final score is transparent and componentized. Geography is computed after matching as a diagnostic component and is not used to choose the product/activity candidate.

In [43]:
SEMANTIC_MODEL = None
SEMANTIC_MODEL_ERROR = None


def get_semantic_model():
    """Load an optional local sentence-transformer model if available."""
    global SEMANTIC_MODEL, SEMANTIC_MODEL_ERROR
    if not SENTENCE_TRANSFORMERS_AVAILABLE:
        return None
    if SEMANTIC_MODEL is not None or SEMANTIC_MODEL_ERROR is not None:
        return SEMANTIC_MODEL
    try:
        SEMANTIC_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
    except Exception as exc:
        SEMANTIC_MODEL_ERROR = str(exc)
        cprint(
            "sentence-transformers is installed, but no usable local model was loaded. "
            "Semantic matching is disabled.",
            "yellow",
        )
        SEMANTIC_MODEL = None
    return SEMANTIC_MODEL


def semantic_score_optional(left, right):
    """Return optional semantic similarity from 0 to 100, or 0 when disabled."""
    model = get_semantic_model()
    if model is None:
        return 0.0
    left_clean = clean_text(left)
    right_clean = clean_text(right)
    if not left_clean or not right_clean:
        return 0.0
    embeddings = model.encode([left_clean, right_clean], convert_to_tensor=True)
    score = util.cos_sim(embeddings[0], embeddings[1]).item()
    return float(max(0.0, min(100.0, score * 100.0)))


def domain_flags(text):
    """Return technical domain labels detected in text."""
    cleaned = clean_text(text)
    flags = set()
    for label, terms in DOMAIN_TERMS.items():
        if any(term in cleaned for term in terms):
            flags.add(label)
    return flags


def context_overlap_score(bafu_text, eco_text):
    """Score overlap in coarse technical domains."""
    bafu_flags = domain_flags(bafu_text)
    eco_flags = domain_flags(eco_text)
    if not bafu_flags and not eco_flags:
        return 50.0
    if not bafu_flags or not eco_flags:
        return 20.0
    overlap = len(bafu_flags & eco_flags)
    union = len(bafu_flags | eco_flags)
    return float(round(100.0 * overlap / union, 2))


def formula_like_terms(text):
    """Extract simple formula-like chemical tokens."""
    cleaned = as_text(text)
    return set(re.findall(r"\b(?:[A-Z][a-z]?\d*){2,}\b", cleaned))


def chemical_identity_score(bafu_row, candidate):
    """Score CAS and formula-like evidence for chemical products."""
    bafu_text = f"{bafu_row['activity name']} {bafu_row['reference product']}"
    eco_text = (
        f"{candidate['Activity Name']} {candidate['Reference Product Name']} "
        f"{candidate.get('Product Information', '')} {candidate.get('CAS Number', '')}"
    )
    if "chemical" not in (domain_flags(bafu_text) | domain_flags(eco_text)):
        return 0.0
    bafu_formulae = formula_like_terms(bafu_text)
    eco_formulae = formula_like_terms(eco_text)
    formula_score = 100.0 if bafu_formulae and bafu_formulae & eco_formulae else 0.0
    cas_text = as_text(candidate.get("CAS Number", ""))
    cas_score = 85.0 if cas_text else 0.0
    return max(formula_score, cas_score)


def term_present(text, terms):
    """Return True when any term appears in normalized text."""
    cleaned = clean_text(text, remove_fillers=False)
    return any(term in cleaned for term in terms)


def detect_context_warnings(bafu_row, candidate, stage, product_fuzzy_score):
    """Detect context mismatches and return warnings plus a penalty."""
    warnings_found = []
    penalty = 0.0
    bafu_text = f"{bafu_row['activity name']} {bafu_row['reference product']}"
    eco_text = (
        f"{candidate['Activity Name']} {candidate['Reference Product Name']} "
        f"{candidate.get('Sector', '')} {candidate.get('Product Information', '')}"
    )

    for _label, terms, warning, term_penalty in CONTEXT_WARNING_RULES:
        if term_present(eco_text, terms) and not term_present(bafu_text, terms):
            warnings_found.append(warning)
            penalty += term_penalty

    if stage == "activity":
        warnings_found.append("activity_only_match")

    if product_fuzzy_score < 70 and stage == "product":
        warnings_found.append("product_not_found")
        penalty += 20.0

    if product_fuzzy_score < 88 and stage != "activity":
        warnings_found.append("broad_proxy_warning")
        penalty += 10.0

    generic_terms = {"product", "service", "goods", "material"}
    product_tokens = set(tokenize(candidate["Reference Product Name"]))
    if product_tokens and product_tokens <= generic_terms:
        warnings_found.append("generic_product_warning")
        penalty += 15.0

    return sorted(set(warnings_found)), float(penalty)


def score_candidate(bafu_row, candidate, stage):
    """Score one ecoinvent candidate against one simplified BAFU row."""
    unit_ok = int(as_text(bafu_row["normalized_bafu_unit"]) == as_text(candidate["_normalized_unit"]))
    product_exact_score = (
        100.0
        if clean_text(bafu_row["reference product"]) == clean_text(candidate["Reference Product Name"])
        else 0.0
    )
    product_fuzzy_score = token_score(bafu_row["reference product"], candidate["Reference Product Name"])
    activity_fuzzy_score = token_score(bafu_row["activity name"], candidate["Activity Name"])
    semantic_score = max(
        semantic_score_optional(bafu_row["reference product"], candidate["Reference Product Name"]),
        semantic_score_optional(bafu_row["activity name"], candidate["Activity Name"]),
    )
    sector_context_score = context_overlap_score(
        f"{bafu_row['activity name']} {bafu_row['reference product']}",
        f"{candidate['Activity Name']} {candidate['Reference Product Name']} {candidate.get('Sector', '')}",
    )
    chemical_score = chemical_identity_score(bafu_row, candidate)
    warnings_found, negative_penalty = detect_context_warnings(
        bafu_row,
        candidate,
        stage,
        product_fuzzy_score,
    )

    if not unit_ok:
        warnings_found.append("unit_mismatch_blocked")
        negative_penalty += 100.0

    product_evidence = max(product_exact_score, product_fuzzy_score)
    if stage == "product":
        raw_score = (
            0.44 * product_evidence
            + 0.24 * activity_fuzzy_score
            + 0.12 * semantic_score
            + 0.10 * sector_context_score
            + 0.10 * chemical_score
        )
    else:
        raw_score = (
            0.18 * product_fuzzy_score
            + 0.48 * activity_fuzzy_score
            + 0.16 * semantic_score
            + 0.10 * sector_context_score
            + 0.08 * chemical_score
        )

    final_score = max(0.0, min(100.0, raw_score - negative_penalty))

    return {
        "activity name": bafu_row["activity name"],
        "reference product": bafu_row["reference product"],
        "activity unit": bafu_row["activity unit"],
        "normalized_bafu_unit": bafu_row["normalized_bafu_unit"],
        "bafu_activity_uuids": bafu_row["bafu_activity_uuids"],
        "bafu_product_uuids": bafu_row["bafu_product_uuids"],
        "bafu_geographies": bafu_row["bafu_geographies"],
        "eco_group_id": int(candidate["eco_group_id"]),
        "matched ecoinvent activity name": candidate["Activity Name"],
        "ecoinvent activity UUID": candidate["Activity UUID"],
        "ecoinvent reference product": candidate["Reference Product Name"],
        "ecoinvent product UUID": candidate["Product UUID"],
        "ecoinvent unit": candidate["Unit"],
        "normalized ecoinvent unit": candidate["_normalized_unit"],
        "all available ecoinvent geographies": candidate["Geography"],
        "ecoQuery URL": candidate["ecoQuery URL"],
        "sector": candidate["Sector"],
        "ISIC classification": candidate["ISIC Classification"],
        "CPC classification": candidate["CPC Classification"],
        "HS2017 classification": candidate["HS2017 Classification"],
        "CAS number": candidate["CAS Number"],
        "product information": candidate["Product Information"],
        "match stage": stage,
        "unit_ok": unit_ok,
        "product_exact_score": round(product_exact_score, 2),
        "product_fuzzy_score": round(product_fuzzy_score, 2),
        "activity_fuzzy_score": round(activity_fuzzy_score, 2),
        "semantic_score": round(semantic_score, 2),
        "geography_score": 0.0,
        "sector_context_score": round(sector_context_score, 2),
        "chemical_identity_score": round(chemical_score, 2),
        "negative_penalty": round(negative_penalty, 2),
        "final_score": round(final_score, 2),
        "warnings": " | ".join(sorted(set(warnings_found))),
    }

## 11. Candidate generation, acceptance rules, and review helpers

In [44]:
def serious_warning_present(warnings_text):
    """Return True when serious context warnings are present."""
    warnings_set = {warning.strip() for warning in as_text(warnings_text).split("|") if warning.strip()}
    return bool(warnings_set & SERIOUS_CONTEXT_WARNINGS)


def score_product_candidates(bafu_row, eco_groups, indexes):
    """Score reference-product-first candidates within the same normalized unit."""
    unit = bafu_row["normalized_bafu_unit"]
    if not unit:
        return pd.DataFrame()

    product_key = clean_text(bafu_row["reference product"])
    choices = indexes["product_choices_by_unit"].get(unit, [])
    if not choices:
        return pd.DataFrame()

    candidate_keys = []
    if product_key in choices:
        candidate_keys.append(product_key)

    fuzzy_hits = process.extract(
        product_key,
        choices,
        scorer=fuzz.token_set_ratio,
        limit=TOP_PRODUCT_CANDIDATES,
    )
    for key, score, _index in fuzzy_hits:
        if score >= 55 and key not in candidate_keys:
            candidate_keys.append(key)

    scored_rows = []
    for key in candidate_keys:
        for group_index in indexes["groups_by_unit_product"].get((unit, key), []):
            scored_rows.append(score_candidate(bafu_row, eco_groups.loc[group_index], "product"))

    if not scored_rows:
        return pd.DataFrame()
    return pd.DataFrame(scored_rows).sort_values(
        ["final_score", "product_exact_score", "product_fuzzy_score", "activity_fuzzy_score"],
        ascending=False,
    )


def score_activity_candidates(bafu_row, eco_groups, indexes):
    """Score activity-name candidates within the same normalized unit."""
    unit = bafu_row["normalized_bafu_unit"]
    if not unit:
        return pd.DataFrame()

    activity_key = clean_text(bafu_row["activity name"])
    choices = indexes["activity_choices_by_unit"].get(unit, [])
    if not choices:
        return pd.DataFrame()

    fuzzy_hits = process.extract(
        activity_key,
        choices,
        scorer=fuzz.token_set_ratio,
        limit=TOP_ACTIVITY_CANDIDATES,
    )
    candidate_keys = [key for key, score, _index in fuzzy_hits if score >= 58]

    scored_rows = []
    for key in candidate_keys:
        for group_index in indexes["groups_by_unit_activity"].get((unit, key), []):
            scored_rows.append(score_candidate(bafu_row, eco_groups.loc[group_index], "activity"))

    if not scored_rows:
        return pd.DataFrame()
    return pd.DataFrame(scored_rows).sort_values(
        ["final_score", "activity_fuzzy_score", "product_fuzzy_score"],
        ascending=False,
    )


def automatic_status(candidate):
    """Return an automatic match status or None if manual review is needed."""
    if candidate is None or int(candidate.get("unit_ok", 0)) != 1:
        return None
    if serious_warning_present(candidate.get("warnings", "")):
        return None
    stage = candidate["match stage"]
    product_exact = float(candidate["product_exact_score"])
    product_fuzzy = float(candidate["product_fuzzy_score"])
    activity_fuzzy = float(candidate["activity_fuzzy_score"])
    final_score = float(candidate["final_score"])

    if stage == "product" and product_exact == 100.0 and activity_fuzzy >= 35 and final_score >= 70:
        return "accepted_exact_product"
    if stage == "product" and product_fuzzy >= PRODUCT_FUZZY_ACCEPT_THRESHOLD and final_score >= 82:
        return "accepted_fuzzy_product"
    if stage == "activity" and final_score >= ACTIVITY_ONLY_ACCEPT_THRESHOLD:
        return "accepted_activity_only"
    return None


def empty_decision(bafu_row, status, warnings_text):
    """Create an empty decision row for unmatched or blocked BAFU records."""
    return {
        "activity name": bafu_row["activity name"],
        "reference product": bafu_row["reference product"],
        "activity unit": bafu_row["activity unit"],
        "normalized_bafu_unit": bafu_row["normalized_bafu_unit"],
        "bafu_activity_uuids": bafu_row["bafu_activity_uuids"],
        "bafu_product_uuids": bafu_row["bafu_product_uuids"],
        "bafu_geographies": bafu_row["bafu_geographies"],
        "eco_group_id": np.nan,
        "matched ecoinvent activity name": "",
        "ecoinvent activity UUID": "",
        "ecoinvent reference product": "",
        "ecoinvent product UUID": "",
        "ecoinvent unit": "",
        "normalized ecoinvent unit": "",
        "all available ecoinvent geographies": "",
        "ecoQuery URL": "",
        "sector": "",
        "ISIC classification": "",
        "CPC classification": "",
        "HS2017 classification": "",
        "CAS number": "",
        "product information": "",
        "match stage": "",
        "match status": status,
        "unit_ok": 0,
        "product_exact_score": 0.0,
        "product_fuzzy_score": 0.0,
        "activity_fuzzy_score": 0.0,
        "semantic_score": 0.0,
        "geography_score": 0.0,
        "sector_context_score": 0.0,
        "chemical_identity_score": 0.0,
        "negative_penalty": 0.0,
        "final_score": 0.0,
        "warnings": warnings_text,
    }


def decision_from_candidate(candidate, status):
    """Create a match decision from a scored candidate row."""
    decision = dict(candidate)
    existing = as_text(decision.get("warnings", ""))
    if status == "needs_manual_review" and "manual_review_required" not in existing:
        existing = " | ".join(sorted({*(warning.strip() for warning in existing.split("|") if warning.strip()), "manual_review_required"}))
    decision["match status"] = status
    decision["warnings"] = existing
    return decision

In [45]:
def interactive_review_candidate(bafu_row, candidates):
    """Prompt the user to choose a candidate, skip, or export the candidate set."""
    if candidates.empty:
        return None

    display_columns = [
        "matched ecoinvent activity name",
        "ecoinvent reference product",
        "ecoinvent unit",
        "all available ecoinvent geographies",
        "match stage",
        "final_score",
        "product_fuzzy_score",
        "activity_fuzzy_score",
        "warnings",
    ]
    cprint("\nAmbiguous BAFU row", "bold yellow")
    print(f"Activity: {bafu_row['activity name']}")
    print(f"Reference product: {bafu_row['reference product']}")
    print(f"Unit: {bafu_row['activity unit']} -> {bafu_row['normalized_bafu_unit']}")
    display(candidates[display_columns].head(MAX_REVIEW_CANDIDATES_PER_ROW).reset_index(drop=True))
    answer = input("Choose candidate number, 's' to skip, or 'e' to export/review later: ").strip().lower()
    if answer == "s":
        return None
    if answer == "e":
        return "export"
    if answer.isdigit():
        choice = int(answer)
        if 0 <= choice < min(MAX_REVIEW_CANDIDATES_PER_ROW, len(candidates)):
            return candidates.iloc[choice].to_dict()
    print("Input not understood. Keeping this row for manual review.")
    return "export"


def export_manual_review_candidates(candidates, output_path):
    """Write ambiguous candidates to Excel for manual review."""
    if candidates.empty:
        pd.DataFrame({"note": ["No manual review candidates were produced."]}).to_excel(output_path, index=False)
    else:
        candidates.to_excel(output_path, index=False)
    print(f"Manual review candidate export written to: {output_path}")


def choose_best_candidate(bafu_row, product_candidates, activity_candidates):
    """Choose the best deterministic candidate according to the staged policy."""
    if not product_candidates.empty:
        best_product = product_candidates.iloc[0].to_dict()
        product_status = automatic_status(best_product)
        if product_status:
            return decision_from_candidate(best_product, product_status), product_candidates

        if (
            float(best_product["product_exact_score"]) == 100.0
            or float(best_product["product_fuzzy_score"]) >= MIN_PRODUCT_REFERENCE_SCORE
        ):
            return decision_from_candidate(best_product, "needs_manual_review"), product_candidates

    if not activity_candidates.empty:
        best_activity = activity_candidates.iloc[0].to_dict()
        activity_status = automatic_status(best_activity)
        if activity_status:
            return decision_from_candidate(best_activity, activity_status), activity_candidates
        if float(best_activity["final_score"]) >= MIN_REVIEW_SCORE:
            return decision_from_candidate(best_activity, "needs_manual_review"), activity_candidates

    return empty_decision(bafu_row, "unmatched", "product_not_found"), pd.DataFrame()


def run_simplified_matching(simplified, eco_groups, indexes, review_mode="export"):
    """Run product-first and activity-second matching over simplified BAFU rows."""
    decisions = []
    manual_review_rows = []
    candidate_pool_rows = []
    counters = Counter()

    iterator = tqdm(simplified.index, total=len(simplified), desc="Matching simplified BAFU")
    for row_index in iterator:
        bafu_row = simplified.loc[row_index]
        if not as_text(bafu_row["normalized_bafu_unit"]):
            decision = empty_decision(bafu_row, "unit_unmapped", "unit_unmapped")
            decisions.append(decision)
            counters["unit_unmapped"] += 1
            continue

        product_candidates = score_product_candidates(bafu_row, eco_groups, indexes)
        if not product_candidates.empty:
            candidate_pool_rows.extend(product_candidates.to_dict("records"))

        activity_candidates = pd.DataFrame()
        needs_activity_stage = product_candidates.empty
        if not product_candidates.empty:
            best_product = product_candidates.iloc[0]
            needs_activity_stage = (
                float(best_product["product_exact_score"]) < 100.0
                and float(best_product["product_fuzzy_score"]) < MIN_PRODUCT_REFERENCE_SCORE
            )

        if needs_activity_stage:
            activity_candidates = score_activity_candidates(bafu_row, eco_groups, indexes)
            if not activity_candidates.empty:
                candidate_pool_rows.extend(activity_candidates.to_dict("records"))

        decision, review_candidates = choose_best_candidate(bafu_row, product_candidates, activity_candidates)

        if decision["match status"] == "needs_manual_review":
            if review_mode == "interactive":
                reviewed = interactive_review_candidate(bafu_row, review_candidates)
                if isinstance(reviewed, dict):
                    decision = decision_from_candidate(reviewed, "accepted_by_user")
                elif reviewed is None:
                    decision = empty_decision(bafu_row, "unmatched", "manual_review_skipped")
            if not review_candidates.empty:
                review_export = review_candidates.head(MAX_REVIEW_CANDIDATES_PER_ROW).copy()
                review_export["manual_review_reason"] = decision.get("warnings", "manual_review_required")
                manual_review_rows.extend(review_export.to_dict("records"))

        decisions.append(decision)
        counters[decision["match status"]] += 1

    decisions_df = pd.DataFrame(decisions)
    manual_df = pd.DataFrame(manual_review_rows).drop_duplicates() if manual_review_rows else pd.DataFrame()
    candidate_pool = pd.DataFrame(candidate_pool_rows).drop_duplicates() if candidate_pool_rows else pd.DataFrame()
    return decisions_df, manual_df, candidate_pool, counters

## 12. Run product-first and activity-second matching

In [46]:
cprint("Stage: product matching and activity fallback", "bold cyan")
if not SENTENCE_TRANSFORMERS_AVAILABLE:
    print("Semantic matching is disabled because sentence-transformers is not installed.")

match_decisions, manual_review_candidates, ecoinvent_candidate_pool, match_counters = run_simplified_matching(
    simplified_bafu,
    eco_groups,
    eco_indexes,
    review_mode=REVIEW_MODE,
)

print("Simplified matching counts:")
for key, value in match_counters.items():
    print(f"  {key}: {value:,}")
print(f"Manual review candidate rows: {len(manual_review_candidates):,}")
print(f"ecoinvent candidate pool rows retained for audit: {len(ecoinvent_candidate_pool):,}")
show_preview(match_decisions, "Simplified match decisions preview")

Stage: product matching and activity fallback
Semantic matching is disabled because sentence-transformers is not installed.


Matching simplified BAFU: 100%|██████████| 7143/7143 [03:33<00:00, 33.44it/s] 


Simplified matching counts:
  needs_manual_review: 3,296
  unmatched: 3,493
  accepted_exact_product: 334
  accepted_fuzzy_product: 20
Manual review candidate rows: 30,763
ecoinvent candidate pool rows retained for audit: 438,083

Simplified match decisions preview


,activity name,reference product,activity unit,normalized_bafu_unit,bafu_activity_uuids,bafu_product_uuids,bafu_geographies,eco_group_id,matched ecoinvent activity name,ecoinvent activity UUID,ecoinvent reference product,ecoinvent product UUID,ecoinvent unit,normalized ecoinvent unit,all available ecoinvent geographies,ecoQuery URL,sector,ISIC classification,CPC classification,HS2017 classification,CAS number,product information,match stage,unit_ok,product_exact_score,product_fuzzy_score,activity_fuzzy_score,semantic_score,geography_score,sector_context_score,chemical_identity_score,negative_penalty,final_score,warnings,match status
0,"1,1-difluoroethane, HFC-152a, at plant","1,1-difluoroethane, HFC-152a, at plant",kilogram,kg,cda7d2ad-0df5-32cc-945b-49d9cb010895,191327,US,2.0,"1,1-difluoroethane production",d62724f9-986a-5c28-8373-171d3201e42c | f2a268ad-c0e6-56c8-abcc-7d38acd8eb36,"1,1-difluoroethane",807906d0-f3cb-4a7c-a528-ae497a61bf12,kg,kg,RoW | US,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/7957/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/d...,Chemicals,2011:Manufacture of basic chemicals,"34110: Hydrocarbons and their halogenated, sulphonated, nitrated or nitrosated derivatives",290371: Halogenated derivatives of acyclic hydrocarbons containing two or more different halogens; chlorodifluoromet...,75-37-6,"'1,1-difluoroethane' is an organic substance with a CAS no. : 000075-37-6. It is called '1,1-difluoroethane' under I...",product,1,0.0,100.00,78.05,0.0,0.0,50.0,85.0,0.0,76.23,manual_review_required,needs_manual_review
1,"1-butanol, propylene hydroformylation, at plant","1-butanol, propylene hydroformylation, at plant",kilogram,kg,b323f136-28be-3e86-bea4-e77610261147,562971,RER,8128.0,"propylene production, from methanol-to-propylene conversion",c42f68a4-63a4-51dd-8dc9-13eb66cdcbaf,propylene,b658a402-79b9-4a02-a818-f6090adb13f1,kg,kg,CN,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/29097/documentation,Chemicals,2011:Manufacture of basic chemicals,"33421: Ethylene, propylene, butylene, butadiene","290122: Acyclic hydrocarbons; unsaturated, propene (propylene)",115-07-1,'propylene' is an organic substance with a CAS no. : 000115-07-1. It is called 'propene' under IUPAC naming and its ...,product,1,0.0,100.00,54.76,0.0,0.0,100.0,85.0,0.0,75.64,manual_review_required,needs_manual_review
2,"1-pentanol, at plant","1-pentanol, at plant",kilogram,kg,7589a6ec-9584-3a34-974b-0ede8a255608,528491,RER,7712.0,"pentanols production, hydroformylation of butene",47884ccd-8296-5ae6-a113-b6a677f37620 | 5197a421-6272-5eb5-863b-3571c50b2c7a,1-pentanol,d3a29af5-314a-4659-a574-6bfd53a6bde0,kg,kg,RER | RoW,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/2855/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/d...,Chemicals,2011:Manufacture of basic chemicals,"34139: Other alcohols, phenols, phenol-alcohols, and their halogenated, sulphonated, nitrated or nitrosated derivati...","290519: Alcohols; saturated monohydric, n.e.c. in item no. 2905.1",71-41-0,'1-pentanol' is an organic substance with a CAS no. : 000071-41-0. It is called 'pentan-1-ol' under IUPAC naming and...,product,1,100.0,100.00,31.58,0.0,0.0,50.0,85.0,0.0,65.08,manual_review_required,needs_manual_review
3,"1-propanol, at plant","1-propanol, at plant",kilogram,kg,183b303d-d07b-3177-b091-c581e9c4ab20,539705,RER,10.0,1-methoxy-2-propanol production,4e19d401-551c-57d5-aebe-99e1dafea51d,1-methoxy-2-propanol,95aedd09-379e-5ee5-b2af-d220968c898b,kg,kg,GLO,https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/25998/documentation,Chemicals,2011:Manufacture of basic chemicals,"34170: Ethers, alcohol peroxides, ether peroxides, epoxides, acetals and hemiacetals, and their halogenated, sulphon...","290943: Ether-alcohols and their halogenated, sulphonated, nitrated or nitrosated derivatives; monobutyl ethers of [...",000107-98-2,'1-methoxy-2-propanol' is an organic substance with the CAS no.: 000107-98-2. It is called '1-methoxypropan-2-ol' u...,product,

## 13. Geography resolution and scientific comments

Geography is resolved after the product/activity decision. A correct activity can remain useful even when ecoinvent lacks the exact BAFU geography, so those rows are retained with an explicit warning.

In [47]:
def split_pipe(value):
    """Split a pipe-separated text field into nonblank values."""
    return [part.strip() for part in as_text(value).split("|") if part.strip()]


def normalize_geography(value):
    """Normalize geography labels for exact comparison."""
    return as_text(value).upper().strip()


def resolve_geography(bafu_geography, available_geographies):
    """Resolve exact geography availability without changing the activity match."""
    bafu_geo = normalize_geography(bafu_geography)
    geographies = split_pipe(available_geographies)
    normalized_lookup = {normalize_geography(geo): geo for geo in geographies}
    if bafu_geo and bafu_geo in normalized_lookup:
        return {
            "ecoinvent geography selected": normalized_lookup[bafu_geo],
            "geography_match_status": "exact_geography_available",
            "geography_score": 100.0,
            "geography_warning": "",
        }
    return {
        "ecoinvent geography selected": "",
        "geography_match_status": "no_exact_geography_available",
        "geography_score": 0.0,
        "geography_warning": "geography_not_available",
    }


def select_ecoinvent_row_for_geography(eco_prepared, eco_group_id, selected_geography):
    """Select the exact-geography ecoinvent row when available, otherwise a representative row."""
    if pd.isna(eco_group_id):
        return None
    rows = eco_prepared.loc[eco_prepared["eco_group_id"] == int(eco_group_id)].copy()
    if rows.empty:
        return None
    selected_norm = normalize_geography(selected_geography)
    if selected_norm:
        exact = rows.loc[rows["Geography"].apply(normalize_geography) == selected_norm]
        if not exact.empty:
            return exact.iloc[0]
    return rows.iloc[0]


def append_warning(warnings_text, warning):
    """Append a warning to a pipe-separated warning field."""
    warnings_set = {part.strip() for part in as_text(warnings_text).split("|") if part.strip()}
    if warning:
        warnings_set.add(warning)
    return " | ".join(sorted(warnings_set))


def make_scientific_comment(row):
    """Explain the scientific basis and limitations of a match decision."""
    status = as_text(row.get("match status", ""))
    if status == "unit_unmapped":
        return "No match was attempted because the BAFU physical unit could not be harmonized to an ecoinvent unit."
    if status == "unmatched":
        return "No acceptable deterministic product or activity candidate was found within the same normalized unit."

    basis = []
    limits = []
    warnings_text = as_text(row.get("warnings", ""))
    bafu_text = f"{row.get('BAFU activity name', '')} {row.get('BAFU reference product', '')}"
    eco_text = (
        f"{row.get('matched ecoinvent activity name', '')} "
        f"{row.get('ecoinvent reference product', '')} "
        f"{row.get('sector', '')}"
    )
    domains = domain_flags(f"{bafu_text} {eco_text}")

    if float(row.get("product_exact_score", 0) or 0) == 100:
        basis.append("identical normalized reference product")
    elif float(row.get("product_fuzzy_score", 0) or 0) >= PRODUCT_FUZZY_ACCEPT_THRESHOLD:
        basis.append("equivalent reference product wording")

    if "activity_only_match" in warnings_text:
        basis.append("activity-name evidence only after no acceptable reference-product match")

    if float(row.get("activity_fuzzy_score", 0) or 0) >= 85:
        basis.append("same or very similar technical activity wording")

    if "chemical" in domains:
        cas = as_text(row.get("CAS number", ""))
        if cas:
            basis.append(f"chemical identity supported by ecoinvent CAS information ({cas})")
        else:
            basis.append("chemical product context based on substance wording; CAS was not available")
        limits.append("chemical purity, compound group, or intermediate status should be checked when the product name is generic")

    if "infrastructure" in domains:
        basis.append("same infrastructure type or infrastructure service function")
        limits.append("infrastructure system boundary can differ between construction, operation, maintenance, and treatment datasets")
    if "energy" in domains:
        basis.append("same energy carrier or energy service function")
    if "transport" in domains:
        basis.append("same transport service context")
    if "waste" in domains:
        basis.append("waste or treatment context is present and must match the intended service function")

    if not basis:
        basis.append("best deterministic technical proxy within the same physical unit")

    if row.get("geography_match_status") == "no_exact_geography_available":
        limits.append("ecoinvent has no exact geography for the BAFU geography; available geographies are listed")
    if warnings_text:
        limits.append(f"warnings: {warnings_text}")
    if as_text(row.get("BAFU activity name", "")) != as_text(row.get("matched ecoinvent activity name", "")):
        limits.append("activity wording differs and should be reviewed if the system boundary is important")

    return "Match basis: " + "; ".join(dict.fromkeys(basis)) + ". Limits: " + "; ".join(dict.fromkeys(limits)) + "."

## 14. Merge simplified decisions back to all non-obsolete BAFU geography rows

In [48]:
FINAL_COLUMNS = [
    "BAFU activity name",
    "BAFU activity UUID",
    "BAFU reference product",
    "BAFU product UUID",
    "BAFU unit",
    "normalized BAFU unit",
    "BAFU geography",
    "matched ecoinvent activity name",
    "ecoinvent activity UUID",
    "ecoinvent reference product",
    "ecoinvent product UUID",
    "ecoinvent unit",
    "normalized ecoinvent unit",
    "ecoinvent geography selected",
    "all available ecoinvent geographies",
    "ecoQuery URL",
    "sector",
    "ISIC classification",
    "CPC classification",
    "HS2017 classification",
    "CAS number",
    "product information",
    "match stage",
    "match status",
    "unit_ok",
    "product_exact_score",
    "product_fuzzy_score",
    "activity_fuzzy_score",
    "semantic_score",
    "geography_score",
    "sector_context_score",
    "chemical_identity_score",
    "negative_penalty",
    "final_score",
    "warnings",
    "geography_match_status",
    "detailed scientific comment",
]


def merge_matches_back_to_bafu(bafu_active, decisions, eco_prepared):
    """Merge simplified match decisions back to every active original BAFU row."""
    key_columns = ["activity name", "reference product", "activity unit"]
    decision_lookup = decisions.set_index(key_columns).to_dict("index")
    output_rows = []

    iterator = tqdm(bafu_active.index, total=len(bafu_active), desc="Merging original BAFU rows")
    for row_index in iterator:
        bafu_row = bafu_active.loc[row_index]
        key = tuple(bafu_row[column] for column in key_columns)
        decision = decision_lookup.get(key)
        if decision is None:
            temp = pd.Series(
                {
                    "activity name": bafu_row["activity name"],
                    "reference product": bafu_row["reference product"],
                    "activity unit": bafu_row["activity unit"],
                    "normalized_bafu_unit": normalize_unit(bafu_row["activity unit"]),
                }
            )
            decision = empty_decision(temp, "unmatched", "product_not_found")

        geo_result = resolve_geography(
            bafu_row["geography"],
            decision.get("all available ecoinvent geographies", ""),
        )
        selected_eco = select_ecoinvent_row_for_geography(
            eco_prepared,
            decision.get("eco_group_id", np.nan),
            geo_result["ecoinvent geography selected"],
        )

        warnings_text = append_warning(decision.get("warnings", ""), geo_result["geography_warning"])
        if decision.get("match status") in {"unmatched", "unit_unmapped"}:
            warnings_text = decision.get("warnings", "")

        output = {
            "BAFU activity name": bafu_row["activity name"],
            "BAFU activity UUID": bafu_row["activity uuid"],
            "BAFU reference product": bafu_row["reference product"],
            "BAFU product UUID": bafu_row["product uuid"],
            "BAFU unit": bafu_row["activity unit"],
            "normalized BAFU unit": decision.get("normalized_bafu_unit", normalize_unit(bafu_row["activity unit"])),
            "BAFU geography": bafu_row["geography"],
            "matched ecoinvent activity name": decision.get("matched ecoinvent activity name", ""),
            "ecoinvent activity UUID": decision.get("ecoinvent activity UUID", ""),
            "ecoinvent reference product": decision.get("ecoinvent reference product", ""),
            "ecoinvent product UUID": decision.get("ecoinvent product UUID", ""),
            "ecoinvent unit": decision.get("ecoinvent unit", ""),
            "normalized ecoinvent unit": decision.get("normalized ecoinvent unit", ""),
            "ecoinvent geography selected": geo_result["ecoinvent geography selected"],
            "all available ecoinvent geographies": decision.get("all available ecoinvent geographies", ""),
            "ecoQuery URL": decision.get("ecoQuery URL", ""),
            "sector": decision.get("sector", ""),
            "ISIC classification": decision.get("ISIC classification", ""),
            "CPC classification": decision.get("CPC classification", ""),
            "HS2017 classification": decision.get("HS2017 classification", ""),
            "CAS number": decision.get("CAS number", ""),
            "product information": decision.get("product information", ""),
            "match stage": decision.get("match stage", ""),
            "match status": decision.get("match status", "unmatched"),
            "unit_ok": decision.get("unit_ok", 0),
            "product_exact_score": decision.get("product_exact_score", 0.0),
            "product_fuzzy_score": decision.get("product_fuzzy_score", 0.0),
            "activity_fuzzy_score": decision.get("activity_fuzzy_score", 0.0),
            "semantic_score": decision.get("semantic_score", 0.0),
            "geography_score": geo_result["geography_score"] if decision.get("match status") not in {"unmatched", "unit_unmapped"} else 0.0,
            "sector_context_score": decision.get("sector_context_score", 0.0),
            "chemical_identity_score": decision.get("chemical_identity_score", 0.0),
            "negative_penalty": decision.get("negative_penalty", 0.0),
            "final_score": decision.get("final_score", 0.0),
            "warnings": warnings_text,
            "geography_match_status": geo_result["geography_match_status"] if decision.get("match status") not in {"unmatched", "unit_unmapped"} else "",
        }

        if selected_eco is not None and output["ecoinvent geography selected"]:
            output["ecoinvent activity UUID"] = selected_eco["Activity UUID"]
            output["ecoQuery URL"] = selected_eco["ecoQuery URL"]
            output["sector"] = selected_eco["Sector"]
            output["ISIC classification"] = selected_eco["ISIC Classification"]
            output["CPC classification"] = selected_eco["CPC Classification"]
            output["HS2017 classification"] = selected_eco["HS2017 Classification"]
            output["CAS number"] = selected_eco["CAS Number"]
            output["product information"] = selected_eco["Product Information"]

        output["detailed scientific comment"] = make_scientific_comment(output)
        output_rows.append(output)

    return pd.DataFrame(output_rows, columns=FINAL_COLUMNS)

In [49]:
cprint("Stage: merging decisions back to full non-obsolete BAFU rows and resolving geography", "bold cyan")
final_matches = merge_matches_back_to_bafu(bafu_active, match_decisions, eco_prepared)
print(f"Final non-obsolete BAFU rows: {len(final_matches):,}")
print("Final match statuses:")
print(final_matches["match status"].value_counts(dropna=False))
print("Geography status for matched rows:")
print(final_matches["geography_match_status"].replace("", np.nan).value_counts(dropna=False))
show_preview(final_matches, "Final matches preview", columns=FINAL_COLUMNS[:14] + ["match status", "final_score", "warnings"])

Stage: merging decisions back to full non-obsolete BAFU rows and resolving geography


Merging original BAFU rows: 100%|██████████| 10458/10458 [00:01<00:00, 5422.37it/s]


Final non-obsolete BAFU rows: 10,458
Final match statuses:
match status
needs_manual_review       5089
unmatched                 4919
accepted_exact_product     429
accepted_fuzzy_product      21
Name: count, dtype: int64
Geography status for matched rows:
geography_match_status
NaN                             4919
no_exact_geography_available    3070
exact_geography_available       2469
Name: count, dtype: int64

Final matches preview


,BAFU activity name,BAFU activity UUID,BAFU reference product,BAFU product UUID,BAFU unit,normalized BAFU unit,BAFU geography,matched ecoinvent activity name,ecoinvent activity UUID,ecoinvent reference product,ecoinvent product UUID,ecoinvent unit,normalized ecoinvent unit,ecoinvent geography selected,match status,final_score,warnings
0,"1,1-difluoroethane, HFC-152a, at plant",cda7d2ad-0df5-32cc-945b-49d9cb010895,"1,1-difluoroethane, HFC-152a, at plant",191327,kilogram,kg,US,"1,1-difluoroethane production",f2a268ad-c0e6-56c8-abcc-7d38acd8eb36,"1,1-difluoroethane",807906d0-f3cb-4a7c-a528-ae497a61bf12,kg,kg,US,needs_manual_review,76.23,manual_review_required
1,"1-butanol, propylene hydroformylation, at plant",b323f136-28be-3e86-bea4-e77610261147,"1-butanol, propylene hydroformylation, at plant",562971,kilogram,kg,RER,"propylene production, from methanol-to-propylene conversion",c42f68a4-63a4-51dd-8dc9-13eb66cdcbaf,propylene,b658a402-79b9-4a02-a818-f6090adb13f1,kg,kg,,needs_manual_review,75.64,geography_not_available | manual_review_required
2,"1-pentanol, at plant",7589a6ec-9584-3a34-974b-0ede8a255608,"1-pentanol, at plant",528491,kilogram,kg,RER,"pentanols production, hydroformylation of butene",5197a421-6272-5eb5-863b-3571c50b2c7a,1-pentanol,d3a29af5-314a-4659-a574-6bfd53a6bde0,kg,kg,RER,needs_manual_review,65.08,manual_review_required
3,"1-propanol, at plant",183b303d-d07b-3177-b091-c581e9c4ab20,"1-propanol, at plant",539705,kilogram,kg,RER,1-methoxy-2-propanol production,4e19d401-551c-57d5-aebe-99e1dafea51d,1-methoxy-2-propanol,95aedd09-379e-5ee5-b2af-d220968c898b,kg,kg,,needs_manual_review,81.50,geography_not_available | manual_review_required
4,"1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof",ec675697-6d86-3ed6-bbb6-430db22a50ee,"1.3 MWp slanted-roof installation, multi-Si, panel, mounted, on roof",369027,unit,unit,CH,"photovoltaic slanted-roof installation, 3kWp, multi-Si, panel, mounted, on roof",e307bf10-441c-5faa-a9a8-f7876c35f8d3,"photovoltaic slanted-roof installation, 3kWp, multi-Si, panel, mounted, on roof",df39a019-ca9d-43a7-a49e-083b3cb118a4,unit,unit,CH,needs_manual_review,65.05,manual_review_required
5,"156 kWp flat-roof installation, multi-Si, on roof",7e86cb3d-68bc-3d47-98d9-4ea7726acc2e,"156 kWp flat-roof installation, multi-Si, on roof",230442,unit,unit,CH,"photovoltaic flat-roof installation, 3kWp, multi-Si, on roof",242f94d2-4313-5830-ad9f-547b459e4dda,"photovoltaic flat-roof installation, 3kWp, multi-Si, on roof",81bc88fb-1dcc-4fef-a420-51351f8b5220,unit,unit,CH,needs_manual_review,62.84,manual_review_required
6,"156 kWp flat-roof installation, single-Si, on roof",7c5bcdfa-391e-388b-a2ec-2d6191d08437,"156 kWp flat-roof installation, single-Si, on roof",282327,unit,unit,CH,"photovoltaic flat-roof installation, 3kWp, single-Si, on roof",bfd20619-76c6-5a1b-9bd1-0930c5f95efa,"photovoltaic flat-roof installation, 3kWp, single-Si, on roof",94e76ec7-dd3c-44f2-acec-8d6ec2907e75,unit,unit,CH,needs_manual_review,63.03,manual_review_required
7,"16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si",f343e106-7a73-3c54-b652-f574d1caea9b,"16.667kWp, 100m2, PV system for PVT, installed on slanted roof, single-Si",332989,unit,unit,CH,,,,,,,,unmatched,0.00,product_not_found


## 15. Build run summary and export all audit outputs

In [50]:
def make_summary_dataframe(summary):
    """Convert a summary dictionary to a two-column dataframe."""
    return pd.DataFrame([{"metric": key, "value": value} for key, value in summary.items()])


def build_run_summary(final_matches, match_decisions, obsolete_bafu, manual_review_candidates):
    """Build machine-readable summary metrics for the audit run."""
    accepted_statuses = {"accepted_exact_product", "accepted_fuzzy_product", "accepted_activity_only", "accepted_by_user"}
    matched_rows = final_matches["match status"].isin(accepted_statuses).sum()
    provisional_rows = final_matches["match status"].eq("needs_manual_review").sum()
    no_geo_rows = final_matches["geography_match_status"].eq("no_exact_geography_available").sum()
    summary = {
        "run_started_at": RUN_STARTED_AT,
        "run_finished_at": datetime.now().isoformat(timespec="seconds"),
        "review_mode": REVIEW_MODE,
        "raw_bafu_rows": int(len(bafu_raw)),
        "non_obsolete_bafu_rows": int(len(bafu_active)),
        "obsolete_bafu_rows_excluded": int(len(obsolete_bafu)),
        "simplified_bafu_rows": int(len(simplified_bafu)),
        "ecoinvent_raw_rows": int(len(eco_raw)),
        "ecoinvent_grouped_candidate_rows": int(len(eco_groups)),
        "accepted_exact_product_simplified_rows": int(match_decisions["match status"].eq("accepted_exact_product").sum()),
        "accepted_fuzzy_product_simplified_rows": int(match_decisions["match status"].eq("accepted_fuzzy_product").sum()),
        "accepted_activity_only_simplified_rows": int(match_decisions["match status"].eq("accepted_activity_only").sum()),
        "manual_review_simplified_rows": int(match_decisions["match status"].eq("needs_manual_review").sum()),
        "unmatched_simplified_rows": int(match_decisions["match status"].eq("unmatched").sum()),
        "unit_unmapped_simplified_rows": int(match_decisions["match status"].eq("unit_unmapped").sum()),
        "matched_bafu_rows_accepted": int(matched_rows),
        "matched_bafu_rows_provisional_needs_review": int(provisional_rows),
        "unmatched_bafu_rows": int(final_matches["match status"].eq("unmatched").sum()),
        "unit_unmapped_bafu_rows": int(final_matches["match status"].eq("unit_unmapped").sum()),
        "matched_rows_without_exact_geography": int(no_geo_rows),
        "manual_review_candidate_rows_exported": int(len(manual_review_candidates)),
        "output_workbook": str(OUTPUT_WORKBOOK),
        "manual_review_candidates_export": str(REVIEW_CANDIDATES_EXPORT),
        "obsolete_export": str(OBSOLETE_EXPORT),
    }
    return summary


def safe_sheet(df, note):
    """Return a non-empty dataframe so every requested workbook sheet exists."""
    if df is None or df.empty:
        return pd.DataFrame({"note": [note]})
    return df


def format_workbook(path):
    """Apply simple readable formatting to an Excel workbook."""
    from openpyxl import load_workbook

    workbook = load_workbook(path)
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter
            max_length = 10
            for cell in column_cells[:200]:
                max_length = max(max_length, min(60, len(as_text(cell.value))))
            worksheet.column_dimensions[column_letter].width = min(60, max(12, max_length + 2))
    workbook.save(path)


def export_audit_workbook(
    output_path,
    final_matches,
    simplified_bafu,
    match_decisions,
    manual_review_candidates,
    obsolete_bafu,
    ecoinvent_candidate_pool,
    run_summary,
):
    """Export the final multi-sheet audit workbook."""
    simplified_with_decisions = simplified_bafu.merge(
        match_decisions,
        on=["activity name", "reference product", "activity unit", "normalized_bafu_unit"],
        how="left",
        suffixes=("", "_match"),
    )
    accepted_matches = final_matches.loc[
        final_matches["match status"].isin(
            ["accepted_exact_product", "accepted_fuzzy_product", "accepted_activity_only", "accepted_by_user"]
        )
    ].copy()
    unmatched_bafu = final_matches.loc[final_matches["match status"].eq("unmatched")].copy()
    unit_unmapped = final_matches.loc[final_matches["match status"].eq("unit_unmapped")].copy()
    run_summary_df = make_summary_dataframe(run_summary)

    sheets = {
        "final_matches": safe_sheet(final_matches, "No final matches were produced."),
        "simplified_bafu": safe_sheet(simplified_with_decisions, "No simplified BAFU rows were produced."),
        "accepted_matches": safe_sheet(accepted_matches, "No accepted matches were produced."),
        "manual_review_candidates": safe_sheet(manual_review_candidates, "No manual review candidates were produced."),
        "unmatched_bafu": safe_sheet(unmatched_bafu, "No unmatched BAFU rows were produced."),
        "obsolete_bafu_activities": safe_sheet(obsolete_bafu, "No obsolete BAFU rows were removed."),
        "unit_unmapped": safe_sheet(unit_unmapped, "No unit-unmapped BAFU rows were blocked."),
        "ecoinvent_candidate_pool": safe_sheet(ecoinvent_candidate_pool, "No ecoinvent candidates were scored."),
        "run_summary": run_summary_df,
    }

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, dataframe in sheets.items():
            dataframe.to_excel(writer, sheet_name=sheet_name, index=False)

    format_workbook(output_path)
    print(f"Audit workbook written to: {output_path}")


def print_run_summary(summary):
    """Print the final concise run summary."""
    cprint("\nFinal concise summary", "bold green")
    print(f"Output folder: {WORK_DIR}")
    print(f"Final audit workbook: {summary['output_workbook']}")
    print(f"BAFU rows matched automatically: {summary['matched_bafu_rows_accepted']:,}")
    print(f"BAFU rows requiring manual review: {summary['matched_bafu_rows_provisional_needs_review']:,}")
    print(f"BAFU rows unmatched: {summary['unmatched_bafu_rows']:,}")
    print(f"Obsolete BAFU rows excluded: {summary['obsolete_bafu_rows_excluded']:,}")
    print(f"Unit-unmapped BAFU rows blocked: {summary['unit_unmapped_bafu_rows']:,}")
    print(f"Matched/provisional rows lacking exact geography: {summary['matched_rows_without_exact_geography']:,}")
    print(f"Run summary JSON: {RUN_SUMMARY_JSON}")

In [51]:
cprint("Stage: exporting review candidates, final workbook, and run summary", "bold cyan")
if REVIEW_MODE == "export":
    export_manual_review_candidates(manual_review_candidates, REVIEW_CANDIDATES_EXPORT)

run_summary = build_run_summary(final_matches, match_decisions, obsolete_bafu, manual_review_candidates)
export_audit_workbook(
    OUTPUT_WORKBOOK,
    final_matches,
    simplified_bafu,
    match_decisions,
    manual_review_candidates,
    obsolete_bafu,
    ecoinvent_candidate_pool,
    run_summary,
)

with open(RUN_SUMMARY_JSON, "w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2, ensure_ascii=False)
print(f"Machine-readable run summary written to: {RUN_SUMMARY_JSON}")

print_run_summary(run_summary)

Stage: exporting review candidates, final workbook, and run summary


/opt/anaconda3/envs/ab_beta/lib/python3.11/site-packages/xlsxwriter/worksheet.py:1303: UserWarning: Ignoring URL 'https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/10658/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/13207/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17498/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17577/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17634/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17637/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17641/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17722/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17725/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17750/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17781/documentation | https://ecoquery.ecoinvent.org/3.12/cutoff/dataset/17840/documentation | https://ec

Manual review candidate export written to: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/manual_review_candidates.xlsx
Audit workbook written to: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/bafu_ecoinvent_activity_matching_audit.xlsx
Machine-readable run summary written to: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/run_summary.json

Final concise summary
Output folder: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu
Final audit workbook: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/bafu_ecoinvent_activity_matching_audit.xlsx
BAFU rows matched automatically: 450
BAFU rows requiring manual review: 5,089
BAFU rows unmatched: 4,919
Obsolete BAFU rows excluded: 1,289
Unit-unmapped BAFU rows blocked: 0
Matched/provisional rows lacking exact geography: 3,070
Run summary JSON: /Users/anishkoyamparambath/Documents/Github/BAFU4WeLOOP/clean bafu/run_summary.json
